# 16c — Counterfactual Search — DomesticDeclarations

Loads pre-mined constraints (from 16a) and pre-trained VAE (from 16b),
then runs REVISED+ counterfactual search. No mining or training happens here.

In [ ]:
import sys
import os
from pathlib import Path

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

# src/ must be on sys.path for torch.load to unpickle event_log_loader classes
src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
# ===== TEST MODE =====
TEST_MODE = False
TEST_N_SEQUENCES = 50

In [ ]:
import torch

# --- Load dataset + prediction model ---
data_path = _current / 'encoded_data' / 'test_philipp' / 'domestic_declarations_all_5_test.pkl'
full_dataset = torch.load(data_path, weights_only=False)

if TEST_MODE:
    dataset = [full_dataset[i] for i in range(min(TEST_N_SEQUENCES, len(full_dataset)))]
    print(f'TEST MODE: Using {len(dataset)} sequences (subset of {len(full_dataset)})')
else:
    dataset = full_dataset

sample = dataset[0]
n_cat = len(sample[0])
n_num = len(sample[1])
seq_len = sample[0][0].shape[0]
print(f'Dataset: {len(dataset)} sequences, {n_cat} cat, {n_num} num, seq_len={seq_len}')

from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

model_path = _current / 'src' / 'notebooks' / 'training_variational_dropout' / 'DomesticDeclarations' / 'DomesticDeclarations_full_grad_norm_4layer.pkl'
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(model_path), dropout=0.0)
model.eval()
print(f'Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters')

# --- TensorDecoder + activity vocabulary ---
from src.interpretability.utils.tensor_decoder import TensorDecoder

decoder = TensorDecoder(full_dataset)

ACTIVITY_FEATURE = 'Activity'
activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
max_idx = max(activity_idx_to_label.keys())
activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]
eos_idx = next(i for i, name in enumerate(activity_names) if name == 'EOS')

print(f'Activity vocabulary ({len(activity_names)}), EOS={eos_idx}')

In [ ]:
# Load pre-mined constraints + pre-trained VAE
import pickle

constraints_pkl_path = _current / 'encoded_data' / 'domestic_declarations_constraints.pkl'
with open(constraints_pkl_path, 'rb') as f:
    constraints_data = pickle.load(f)

all_constraints = constraints_data['all']
data_conditions = constraints_data['data_conditions']
print(f'Constraints: {len(all_constraints)} (from {constraints_pkl_path.name})')

vae_path = str(_current / 'encoded_data' / 'domestic_declarations_vae.pkl')
print(f'VAE path: {vae_path}')

In [ ]:
from src.interpretability.perturbation_methods import RevisedPlus, RevisedPlusConfig, create_revised_plus_for_model

device = 'mps' if torch.backends.mps.is_available() else 'cpu'

config = RevisedPlusConfig(
    vae_encoder_hidden=50,
    vae_decoder_hidden=50,
    vae_latent_dim=20,
    vae_num_layers=2,
    vae_dropout=0.3,
    vae_kl_weight=0.1,
    n_candidates_per_round=200,
    n_search_rounds=5,
    top_k=5,
    min_plausibility=0.0,
    device=device,
    activity_feature=ACTIVITY_FEATURE,
)

# Create RevisedPlus with pre-mined constraints and pre-trained VAE
# No mining, no training — just loads both
rp = create_revised_plus_for_model(
    model=model,
    dataset=dataset,
    activity_names=activity_names,
    config=config,
    vae_path=vae_path,
    constraints=all_constraints,
    data_conditions=data_conditions,
)
print(f'\nREVISED+ ready (no mining, no training):')
print(f'  VAE parameters: {sum(p.numel() for p in rp.vae.parameters()):,}')
print(f'  All constraints: {len(rp.all_constraints)}')
print(f'  Prefix-safe constraints: {len(rp.prefix_safe_constraints)}')

In [ ]:
import numpy as np
import pandas as pd

# === Scan dataset for candidate sequences ===
N_CANDIDATES = 50
scan_limit = min(500, len(dataset))

candidates = []
seen_cases = set()
for i in range(scan_limit):
    cat_t, num_t, case_id = dataset[i]
    act = cat_t[0]

    # Deduplicate by case (dataset has multiple prefix lengths per case)
    if case_id in seen_cases:
        continue
    seen_cases.add(case_id)

    # Full trace length (non-padding events)
    trace_len = int((act != 0).sum().item())
    act_seq = [activity_names[a.item()] for a in act if a.item() != 0]

    candidates.append({
        'dataset_idx': i,
        'case_id': case_id,
        'trace_len': trace_len,
        'activities': ' -> '.join(act_seq),
    })

    if len(candidates) >= N_CANDIDATES:
        break

df_candidates = pd.DataFrame(candidates)
print(f'Found {len(candidates)} unique cases (scanned {scan_limit})')
print('Set SELECTED_ROW and PREFIX_LEN below to pick a case and prefix length.\n')
display(df_candidates)

In [ ]:
# ============================================
# SELECT A CASE AND PREFIX LENGTH
# ============================================
SELECTED_ROW = 0   # row index in the candidates table above
PREFIX_LEN = 3     # how many events to use as prefix (1 .. trace_len)

# --- Load and truncate to prefix ---
selected = df_candidates.iloc[SELECTED_ROW]
test_idx = selected['dataset_idx']
trace_len = selected['trace_len']
cat_tuple_full, num_tuple_full, case_id = dataset[test_idx]

# Clamp prefix length
prefix_len = max(1, min(PREFIX_LEN, trace_len))
if prefix_len != PREFIX_LEN:
    print(f'Note: PREFIX_LEN clamped to {prefix_len} (trace has {trace_len} events)')

# Build left-padded prefix tensors (keep only first `prefix_len` events)
pad_len = seq_len - prefix_len
cat_tuple = []
for c in cat_tuple_full:
    t = torch.zeros_like(c)
    # Original is left-padded: real events start at (seq_len - trace_len)
    src_start = seq_len - trace_len
    t[pad_len:] = c[src_start:src_start + prefix_len]
    cat_tuple.append(t)
cat_tuple = tuple(cat_tuple)

num_tuple = []
for n in num_tuple_full:
    t = torch.zeros_like(n)
    src_start = seq_len - trace_len
    t[pad_len:] = n[src_start:src_start + prefix_len]
    num_tuple.append(t)
num_tuple = tuple(num_tuple)

# Show what the model predicts for this prefix
prefix_acts = [activity_names[cat_tuple[0][j].item()] for j in range(pad_len, seq_len)]
full_acts = [activity_names[a.item()] for a in cat_tuple_full[0] if a.item() != 0]

cat_in = [c.unsqueeze(0) for c in cat_tuple]
num_in = [n.unsqueeze(0) for n in num_tuple]
with torch.no_grad():
    preds = model((cat_in, num_in))[0]
    logits = preds[0][f'{ACTIVITY_FEATURE}_mean'][0]
    p = torch.softmax(logits, dim=-1)
    top_p, top_idx = p.max(dim=-1)

print(f'Case: {case_id}')
print(f'Full trace ({trace_len}): {" -> ".join(full_acts)}')
print(f'Prefix ({prefix_len}/{trace_len}):  {" -> ".join(prefix_acts)}')
print(f'Predicted next: {activity_names[top_idx.item()]} (p={top_p.item():.3f})')
print()
df_orig = decoder.decode_sequence(cat_tuple, num_tuple, case_id=case_id)
display(df_orig)

In [ ]:
# === Run counterfactual search ===
cat_tensors = [c.unsqueeze(0) for c in cat_tuple]
num_tensors = [n.unsqueeze(0) for n in num_tuple]

explanation = rp.explain(cat_tensors, num_tensors, target_class=None)
print(explanation)

In [ ]:
# === Display all counterfactuals ===
if not explanation.counterfactuals:
    print('No counterfactuals found. Try increasing n_search_rounds or noise_scale.')
else:
    # --- Summary table ---
    rows = []
    for i, cf in enumerate(explanation.counterfactuals):
        rows.append({
            'rank': i + 1,
            'prediction': cf.counterfactual_prediction_name,
            'probability': f'{cf.counterfactual_probability:.3f}',
            'prefix_len': len(cf.activity_sequence),
            'activities': ' -> '.join(activity_names[a] for a in cf.activity_sequence),
            'proximity': f'{cf.proximity:.2f}',
            'sparsity': cf.sparsity,
            'feasibility': f'{cf.feasibility:.3f}',
            'plaus_def': f'{cf.plausibility_definite:.2f}',
            'plaus_opt': f'{cf.plausibility_optimistic:.2f}',
            'score': f'{cf.combined_score:.4f}',
        })

    print('=' * 100)
    print(f'ORIGINAL PREFIX  —  Case: {case_id}')
    print(f'  Activities: {" -> ".join(activity_names[a] for a in explanation.original_activity_sequence)}')
    print(f'  Prediction: {explanation.original_prediction_name} (p={explanation.original_probability:.3f})')
    print(f'  Prefix length: {explanation.prefix_len} events')
    print('=' * 100)
    display(df_orig)

    print(f'\n{"=" * 100}')
    print(f'ALL COUNTERFACTUALS ({len(explanation.counterfactuals)})')
    print('=' * 100)
    display(pd.DataFrame(rows))

    # --- Detailed view of each counterfactual ---
    for i, cf in enumerate(explanation.counterfactuals):
        print(f'\n{"—" * 100}')
        print(f'COUNTERFACTUAL #{i+1}')
        print(f'  Prediction: {cf.counterfactual_prediction_name} (p={cf.counterfactual_probability:.3f})')
        cf_prefix_len = len(cf.activity_sequence)
        print(f'  Prefix length: {cf_prefix_len} events (delta={cf_prefix_len - explanation.prefix_len:+d})')
        print(f'  Activities: {" -> ".join(activity_names[a] for a in cf.activity_sequence)}')
        print(f'  Proximity: {cf.proximity:.3f}  |  Sparsity: {cf.sparsity}  |  Feasibility: {cf.feasibility:.3f}')
        print(f'  Plausibility (definite): {cf.plausibility_definite:.2f}  |  Plausibility (optimistic): {cf.plausibility_optimistic:.2f}')
        print(f'  Combined score: {cf.combined_score:.4f}')

        cf_cat = tuple(cf.cat_sequence)
        cf_num = tuple(cf.num_sequence[:, j] for j in range(cf.num_sequence.shape[1])) if cf.num_sequence is not None else num_tuple
        df_cf = decoder.decode_sequence(cf_cat, cf_num, skip_padding=False)
        display(df_cf)